# Unavailable borrower size in SCR.data — corporate credit, Paraná

How much corporate credit is granted without company revenue on file, broken down
by type of lending institution, from January 2019 to May 2026.

**Source:** SCR.data, Central Bank of Brazil (ODbL licence).
**Methodological decisions:** see `DECISIONS.md` in the repository root.

Working notebook. Consolidated conclusions go to the README.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '../src')   # style.py and ingest.py live in src/

import style          # portfolio visual identity — see style.py
style.apply()

PARQUET = '../data/processed/scrdata_pj_br.parquet'   # nationwide

# Source columns are in Portuguese (Central Bank of Brazil). They are renamed on
# load so the rest of the notebook reads in one language. The mapping is kept
# explicit rather than hidden in the ingestion step: it documents the origin.
COLUMNS = {
    'data_base':             'month',
    'segmento':              'segment',
    'carteira_ativa':        'outstanding',       # active portfolio, BRL
    'porte_indisponivel':    'size_unavailable',  # borrower size not reported
    'numero_de_operacoes':   'contracts',
    # count withheld for confidentiality
    'operacoes_suprimidas':  'contracts_masked',
}

SEGMENTS = {
    'Banco':                     'Bank',
    'Fintech':                   'Fintech',
    'Instituição de pagamento':  'Payment institution',
}

STUDIED = list(SEGMENTS.values())

ModuleNotFoundError: No module named 'estilo'

## 1. Load and sanity-check the extract

Before any analysis: is the period complete, and does the volume match what the
ingestion routine reported?

In [ ]:
# Only the columns the analysis uses: the nationwide extract is ~11.3M
# rows, and the full column set does not fit comfortably in memory.
df = (pd.read_parquet(PARQUET, columns=list(COLUMNS))
        .rename(columns=COLUMNS)
        .assign(segment=lambda d: d['segment'].map(SEGMENTS).fillna(d['segment'])))

print(f'rows     : {len(df):,}')
print(f'period   : {df["month"].min():%m-%Y} to {df["month"].max():%m-%Y}')
print(f'months   : {df["month"].nunique()}')
print(f'segments : {df["segment"].nunique()}')

## 2. Who is who in the state's credit market

Before comparing proportions across segments, we need to know how big each one
is. A proportion without volume beside it is misleading.

In [ ]:
monthly_total = df.groupby('month', observed=True)['outstanding'].sum()

by_segment = (df.groupby(['month', 'segment'], observed=True)['outstanding']
                .sum().unstack('segment'))

market_share = by_segment.div(monthly_total, axis=0)

summary = pd.DataFrame({
    'median_outstanding_BRL': by_segment.median(),
    'median_share_%': (market_share.median() * 100).round(3),
    'max_share_%': (market_share.max() * 100).round(3),
})
summary.sort_values('median_share_%', ascending=False)

### Reading

Banks dominate credit in the state. Fintechs and payment institutions are very
small slices of the total.

**Consequence for the thesis:** any high proportion found in those two segments
is a large slice of a small whole. That does not invalidate the finding, but it
has to be stated alongside it.

## 3. The decisive test — unavailable size within each segment

The original hypothesis was that traditional banks cannot classify a meaningful
share of small companies. If that were true, the share of unavailable size
*within* the Bank segment would have to be high.

Measured on `outstanding` — see `DECISIONS.md` for why.

In [ ]:
latest = df[df['month'] == df['month'].max()]

test = (latest.groupby(['segment', 'size_unavailable'], observed=True)['outstanding']
              .sum().unstack('size_unavailable', fill_value=0))
test.columns = ['size_reported', 'size_unavailable']
test['unavailable_share_%'] = (
    test['size_unavailable'] / test.sum(axis=1) * 100).round(1)

test.sort_values('unavailable_share_%', ascending=False)

### Reading

The original hypothesis **does not hold**: banks know the revenue of almost
everything they lend to.

What replaces it: the empty field describes **the method of whoever granted the
credit**, not the borrower. The question becomes how much credit is granted
without looking at revenue, and how fast that is growing.

## 4. The monthly series, unfiltered

The raw chart first, with everything in it. It shows both the finding and the
problem.

In [ ]:
base = df[df['segment'].isin(STUDIED)]

# Denominator: all active credit per month and segment.
total = base.groupby(['month', 'segment'], observed=True)['outstanding'].sum()

# Numerator: the slice with no reported borrower size.
# reindex + fill_value=0 matters: a month with no such credit is a true zero,
# not a gap. Without it the line breaks and reads as missing data.
unavailable = (base[base['size_unavailable']]
               .groupby(['month', 'segment'], observed=True)['outstanding'].sum()
               .reindex(total.index, fill_value=0))

series = (unavailable / total).unstack('segment')

fig, ax = style.figure('Unavailable borrower size as a share of outstanding credit',
                        'Share of outstanding',
                        subtitle='Corporate credit, Paraná — raw series, no stability filter')
series.plot(ax=ax, color=style.colors(series.columns))
style.finalize(ax, 'Source: SCR.data, Central Bank of Brazil (ODbL)')
plt.show()

### Reading

Two lines behave and one does not.

- **Bank**: flat at around 2% for seven years. It crosses the pandemic and a full
  interest-rate cycle without moving. It is close to a perfect control group.
- **Fintech**: starts near zero and rises steadily. A trend, not noise.
- **Payment institution**: goes to 97%, drops to zero, comes back to 37%. That is
  not economic behaviour.

The unstable stretch sits at the start of the series, when those segments barely
existed in the state. A proportion computed over a tiny portfolio moves in full
with very few contracts.

## 5. Diagnosis: what makes a proportion unstable

The cut-off criterion went through two versions. Both are recorded here, because
the reason for each change is part of the method.

**First attempt — a floor on market share.** Discarded. Fintechs are even smaller
than payment institutions, so any cut based on relative size would drop both — and
would drop fintechs for the wrong reason: being small is not the same as being
unstable.

**Second attempt — a stability criterion.** The cut targets fragility itself: how
much a single average-sized contract can shift the proportion in a given month.
Where that displacement is large, the line is not trustworthy, regardless of how
big the segment is.

In [ ]:
# `contracts` is withheld whenever a cell holds few operations.
# Dropping those rows understates the total — and understates it MORE in small
# segments, which are exactly the ones this thesis observes. The filter would be
# biased against fintechs for the wrong reason.
#
# Instead of dropping: each masked cell counts as at least one contract.

known = base['contracts'].where(~base['contracts_masked'], 0)

diag = (base.assign(_known=known)
            .groupby(['month', 'segment'], observed=True)
            .agg(outstanding=('outstanding', 'sum'),
                 known_contracts=('_known', 'sum'),
                 masked_cells=('contracts_masked', 'sum'),
                 cells=('contracts_masked', 'size')))

diag['min_contracts'] = diag['known_contracts'] + diag['masked_cells']
diag['masked_share'] = diag['masked_cells'] / diag['cells']

# Displacement caused by one average-sized contract, in percentage points.
#
# It is a CEILING with respect to the arithmetic: min_contracts is a floor on the
# contract count, so 100 / min_contracts is the largest value this ratio can take.
# It is a FLOOR with respect to reality: value is more concentrated than count, so
# one large contract shifts the proportion by more than 1/N. The metric bounds the
# average case, not the worst case.
diag['max_shift_pp'] = 100 / diag['min_contracts']

diag.groupby('segment', observed=True)[
    ['max_shift_pp', 'masked_share']].describe().round(3)

In [ ]:
shift = diag['max_shift_pp'].unstack('segment')

CEILING_PP = 0.1   # see DECISIONS.md — equivalent to requiring ~1,000 contracts

fig, ax = style.figure('Maximum shift caused by one average-sized contract',
                        'percentage points',
                        subtitle='Log scale — the higher the line, the more fragile the proportion')
shift.plot(ax=ax, logy=True, color=style.colors(shift.columns))
ax.axhline(CEILING_PP, color=style.YELLOW,
           linestyle='--', linewidth=1.2, alpha=0.9)
ax.text(shift.index[0], CEILING_PP * 1.08, ' ceiling = 0.1 pp (~1,000 contracts)',
        color=style.YELLOW, fontsize=8.5, va='bottom')
style.finalize(ax, 'Source: SCR.data, Central Bank of Brazil (ODbL)')
plt.show()

### Reading

The Bank line sits at roughly 0.00007 pp across the whole series — on the order of
1.5 million contracts a month. It was never at risk of instability, which confirms
its role as a control group through an independent route.

The Fintech line spans almost four orders of magnitude. Early in the series the
shift approaches 50 pp, which means `min_contracts = 2`: **two contracts**. A single
contract moved half the proportion. From 2022 onwards the line settles.

The Fintech spike in mid-2024 passes just under the ceiling. A jump in this metric
means a **drop in contract count** — it is the same June 2024 discontinuity already
listed under next steps. The stability diagnosis found on its own the break that
was queued for investigation.

In [ ]:
eligible = shift <= CEILING_PP
filtered = series.where(eligible[series.columns])

fig, ax = style.figure('Unavailable borrower size as a share of outstanding credit',
                        'Share of outstanding',
                        subtitle='Corporate credit, Paraná — months passing the stability criterion only')
filtered.plot(ax=ax, color=style.colors(filtered.columns))
style.finalize(ax, 'Source: SCR.data, Central Bank of Brazil (ODbL)')
plt.show()

print('months kept per segment:')
print(eligible[series.columns].sum())

### What the filter fixed, and what it did not

**Fixed for fintechs.** The 2019–2021 stretch drops out, and what remains still
climbs from near zero to about 0.31 — using months that passed the criterion. The
thesis survives its own filter, and comes out stronger: the rise no longer depends
on months with a tiny denominator.

**Not fixed for payment institutions.** The line keeps swinging between 0.03 and
0.57 even in months that passed. The reason is a **mismatch of units**: the criterion
is denominated in contract counts, while the series is denominated in outstanding
value. With a thousand contracts and value concentrated in a few large ones, the
proportion in BRL stays fragile while the metric reports it as safe. The comment in
the previous cell already anticipated this — value is more concentrated than count.

Lowering the ceiling does not fix it: it would drop fintechs along the way, again
for the wrong reason. A criterion denominated in value — for instance, the share of
the segment's portfolio concentrated in its largest cell that month — is the next
step.

**Decision:** payment institutions are **not analysable in value terms** within the
Paraná extract. This is declared as a limitation rather than hidden behind a filter
that does not reach it. The thesis is bank versus fintech. The segment may become
analysable again in the nationwide aggregation, where the denominator is far larger.

## 6. Next steps

- Run the nationwide aggregation. The Paraná extract leaves payment institutions
  without a sufficient denominator and prevents any read by state.
- Test the V1→V2 methodology transition in the source. If the fintech rise coincides
  with the migration, the finding may be a collection artefact — this is a veto, not
  a caveat.
- Investigate the June 2024 break, already located by the stability diagnosis.
- Evaluate a stability criterion denominated in value (portfolio concentration in the
  largest cell of the month), which would reach what the count-based criterion cannot.
- Break down by industry: where the no-revenue method is growing fastest.